# NB_hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` (one row per hospital per day with lab activity) from silver lab results, hospitals and calendar, including a day-over-day LAG trend, per the Silver-to-Gold STTM. Full rebuild via CREATE OR REPLACE TABLE ... AS SELECT.

In [0]:
# Catalog placeholder from the STTM (<catalog>) is exposed as a widget
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog")

src = f"{catalog}.silver"
tgt_table = f"{catalog}.gold.hospital_daily_summary"
print(f"Source schema: {src} | Target table: {tgt_table}")

In [0]:
# Full rebuild: driving set = silver.lab_results grouped by (hospital_id, test_date)
# LEFT JOIN hospitals and calendar so no aggregated row is ever dropped.
# LAG is over the previous summary_date row for the hospital (previous day with lab activity).
spark.sql(f"""
CREATE OR REPLACE TABLE {tgt_table} AS
WITH daily AS (
  SELECT hospital_id,
         test_date                                               AS summary_date,
         COUNT(*)                                                AS total_tests,
         COUNT(DISTINCT patient_id)                              AS unique_patients_tested,
         COUNT(DISTINCT doctor_id)                               AS active_doctor_count,
         COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
         COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count
  FROM {src}.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT d.*,
         LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                   AS hospital_id,
  CAST(h.hospital_name AS STRING)                              AS hospital_name,
  CAST(w.summary_date AS DATE)                                 AS summary_date,
  CAST(c.day_name AS STRING)                                   AS day_name,
  CAST(c.is_weekend AS BOOLEAN)                                AS is_weekend,
  CAST(w.total_tests AS INT)                                   AS total_tests,
  CAST(w.unique_patients_tested AS INT)                        AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                           AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                           AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                           AS critical_test_count,
  CAST(ROUND(w.abnormal_test_count / w.total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(w.critical_test_count / w.total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                          AS prev_day_total_tests,
  CAST(CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
            ELSE ROUND((w.total_tests - w.prev_day_total_tests) / w.prev_day_total_tests * 100, 2)
       END AS DECIMAL(6,2))                                    AS test_volume_change_pct
FROM with_lag w
LEFT JOIN {src}.hospitals h ON w.hospital_id = h.hospital_id
LEFT JOIN {src}.calendar  c ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
""")
print(f"{tgt_table} rebuilt.")

In [0]:
record_count = spark.table(tgt_table).count()
dbutils.notebook.exit(str(record_count))

In [0]:
%sql
select * from adb_caresync.gold.hospital_daily_summary limit 40